## Imports

In [30]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

## Data Load

In [31]:
df = pd.read_csv("../model/placementdata.csv")
df.head()

,StudentID,CGPA,Internships,Projects,Workshops/Certifications,AptitudeTestScore,SoftSkillsRating,ExtracurricularActivities,PlacementTraining,SSC_Marks,HSC_Marks,PlacementStatus
0,1,7.5,1,1,1,65,4.4,No,No,61,79,NotPlaced
1,2,8.9,0,3,2,90,4.0,Yes,Yes,78,82,Placed
2,3,7.3,1,2,2,82,4.8,Yes,No,79,80,NotPlaced
3,4,7.5,1,1,2,85,4.4,Yes,Yes,81,80,Placed
4,5,8.3,1,2,2,86,4.5,Yes,Yes,74,88,Placed


## Basic Cleaning

In [32]:
# drop useless column
df.drop(columns=["StudentID"], inplace=True)

# check nulls
print(df.isnull().sum())

CGPA                         0
Internships                  0
Projects                     0
Workshops/Certifications     0
AptitudeTestScore            0
SoftSkillsRating             0
ExtracurricularActivities    0
PlacementTraining            0
SSC_Marks                    0
HSC_Marks                    0
PlacementStatus              0
dtype: int64


## Outliers

In [33]:
def remove_outliers(df):
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df = df[(df[col] >= lower) & (df[col] <= upper)]
    return df

df = remove_outliers(df)

## Encoder

In [34]:
from sklearn.preprocessing import LabelEncoder

# Target
if df["PlacementStatus"].dtype == "object":
    df["PlacementStatus"] = LabelEncoder().fit_transform(df["PlacementStatus"])

# Features
df["ExtracurricularActivities"] = LabelEncoder().fit_transform(df["ExtracurricularActivities"])
df["PlacementTraining"] = LabelEncoder().fit_transform(df["PlacementTraining"])

In [35]:
df["PlacementTraining"]

0       0
2       0
3       1
4       1
6       0
       ..
9987    1
9988    1
9993    1
9995    0
9997    1
Name: PlacementTraining, Length: 5510, dtype: int64

## Features && Target

In [36]:
X = df.drop("PlacementStatus", axis=1)
y = df["PlacementStatus"]

## Train-Test Split

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Scaling

In [38]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Model Train

In [39]:
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

## Evaluation

In [40]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8003629764065335

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.87      0.86       744
           1       0.71      0.65      0.68       358

    accuracy                           0.80      1102
   macro avg       0.77      0.76      0.77      1102
weighted avg       0.80      0.80      0.80      1102


Confusion Matrix:
 [[649  95]
 [125 233]]


## Sample

In [42]:
sample = pd.DataFrame([[8.5, 2, 4, 2, 85, 4.5, 2, 1, 85, 90]],
                      columns=X.columns)

sample = scaler.transform(sample)

print("Prediction:", model.predict(sample))
print("Probability:", model.predict_proba(sample))

Prediction: [1]
Probability: [[0.05314322 0.94685678]]


## Save Model + Scaler

In [43]:
pickle.dump(model, open("../model/placement_model.pkl", "wb"))
pickle.dump(scaler, open("../model/scaler.pkl", "wb"))